### PCG parameter binary to hex

In [18]:
from pathlib import Path
import math

def bin2hex(input_file, input_format, output_format, output_file):
    in_path = Path(input_file)
    out_path = Path(output_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with open(in_path, 'r', encoding='utf-8') as f:
            lines = f.read().splitlines()

        base_map = {'binary': 2, 'hex': 16, 'decimal': 10}
        results = []
        
        for val in lines:
            val = val.strip()
            if not val: continue
            
            # 1. 取得數值
            decimal_val = int(val, base_map[input_format.lower()])
            
            # 2. 動態判斷補位長度 (Padding)
            # 以輸入字串的長度作為基準
            input_len = len(val)
            
            if output_format.lower() == 'binary':
                # 保持原始輸入長度，不足補 0
                results.append(format(decimal_val, f'0{input_len}b'))
                
            elif output_format.lower() == 'hex':
                # 如果是 binary 轉 hex，通常 4 bits 轉 1 位 hex
                # 使用 math.ceil 確保不足 4 bits 也會佔 1 位
                padding = math.ceil(input_len / 4) if input_format == 'binary' else 2
                results.append(format(decimal_val, f'0{padding}X'))
                
            elif output_format.lower() == 'decimal':
                results.append(str(decimal_val))

        with open(out_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(results))
            
    except Exception as e:
        print(f"檔案 {in_path.name} 轉換失敗: {e}")

def bin2hex_batch(src_dir, dest_dir, in_fmt, out_fmt):
    src_path = Path(src_dir)
    # 支援大小寫 .txt 搜尋
    files = list(src_path.glob('*.txt'))
    
    for f in files:
        target_file_path = Path(dest_dir) / f.name
        bin2hex(str(f), in_fmt, out_fmt, str(target_file_path))
        print(f"已轉換: {f.name} -> {target_file_path.name}")

In [19]:
# Data directory: ./Data_ori/weight -> ./Data_NPU/Data_ori_BtoH
# Data format: Binary -> Hexcimal
bin2hex_batch(
    src_dir='./Data_ori/weight/',
    dest_dir='./Data_NPU/Data_ori_BtoH/',
    in_fmt='binary',
    out_fmt='hex'
)

已轉換: conv1.txt -> conv1.txt
已轉換: conv1_bias.txt -> conv1_bias.txt
已轉換: conv2.txt -> conv2.txt
已轉換: conv2_bias.txt -> conv2_bias.txt
已轉換: conv3.txt -> conv3.txt
已轉換: conv3_bias.txt -> conv3_bias.txt
已轉換: conv4.txt -> conv4.txt
已轉換: conv4_bias.txt -> conv4_bias.txt
已轉換: linear2.txt -> linear2.txt
已轉換: linear2_bias.txt -> linear2_bias.txt
已轉換: scale.txt -> scale.txt
已轉換: zero.txt -> zero.txt


### PCG input rerange

In [33]:
import os

def rearrange_input(source_data, target_data, ch_in, length, word_size=32, data_bit=8):
    """
    連續打包模式：不補 0，資料由右往左 (LSB-first) 填滿 Word 後換行。
    
    :param source_data: 來源 txt 路徑
    :param target_data: 輸出 txt 路徑
    :param ch_in: Channel 數量
    :param length: 每個 Channel 的筆數 (Data size = length * ch_in)
    :param word_size: 一行輸出的總位元數 (預設 32)
    :param data_bit: 單個資料的位元大小 (預設 8)
    """
    # 每個 Word 包含幾個資料位 (例如 32/8 = 4 slots)
    slots_per_word = word_size // data_bit
    # 每個資料點佔幾個 hex 字元 (例如 8-bit = 2 hex chars)
    chars_per_data = data_bit // 4
    
    elements_per_ch = length
    raw_elements = []

    # 確保輸出目錄存在
    os.makedirs(os.path.dirname(target_data), exist_ok=True)

    try:
        # 1. 讀取並攤平所有原始資料
        with open(source_data, 'r', encoding='utf-8') as f:
            content = f.read().split()
            full_text = "".join(content)
            for i in range(0, len(full_text), chars_per_data):
                raw_elements.append(full_text[i:i+chars_per_data])

        # 2. 驗證總量並擷取正確範圍
        expected_total = ch_in * elements_per_ch
        if len(raw_elements) < expected_total:
            print(f"Error: 資料不足。預期 {expected_total} 筆, 實際 {len(raw_elements)} 筆")
            return
        
        raw_elements = raw_elements[:expected_total]

        # 3. 建立 Interleaved 資料流 (A0, B0, C0, D0, A1, B1...)
        data_stream = []
        channels = [raw_elements[c*elements_per_ch : (c+1)*elements_per_ch] for c in range(ch_in)]
        
        for i in range(elements_per_ch):
            for c in range(ch_in):
                data_stream.append(channels[c][i])

        # 4. 依照 Word 寬度進行「由右往左」打包寫入
        with open(target_data, 'w', encoding='utf-8') as f_out:
            for i in range(0, len(data_stream), slots_per_word):
                chunk = data_stream[i : i + slots_per_word]
                
                # 若最後一行的資料點不足一個 word 寬度，補 0
                while len(chunk) < slots_per_word:
                    chunk.append("0" * chars_per_data)
                
                # 關鍵修改：由右往左擺放 (LSB-first)
                # 將 chunk 內容反轉後拼接。例如 [A0, B0, A1, B1] -> B1A1B0A0
                packed_word = "".join(reversed(chunk)) 
                f_out.write(packed_word + '\n')

        print(f"轉換成功：{target_data}")
        print(f"模式：連續打包 (LSB-first), 一行 {word_size}-bit")

    except FileNotFoundError:
        print(f"Error: 找不到檔案 {source_data}")
    except Exception as e:
        print(f"發生錯誤: {e}")

# --- 呼叫測試 ---
# rearrange_input(source_data="input.txt", target_data="output.txt", ch_in=4, length=272)

In [43]:
# conv1 input
rearrange_input("./Data_ori/gold/test_input.txt", "./Data_NPU/model/conv1/ifm.txt", ch_in=1, length=540, word_size=32, data_bit=8)

轉換成功：./Data_NPU/model/conv1/ifm.txt
模式：連續打包 (LSB-first), 一行 32-bit


In [44]:
# conv2 input
# 原本data數量269補至272 (補x)
rearrange_input("./Data_ori/gold/conv1_result.txt", "./Data_NPU/model/conv2/ifm.txt", ch_in=4, length=272, word_size=32, data_bit=8)

轉換成功：./Data_NPU/model/conv2/ifm.txt
模式：連續打包 (LSB-first), 一行 32-bit


### PCG parameter rerange

In [37]:
import os

def rearrange_weight(source_data, target_data, k_size, ch_in, ch_out, word_size=32, data_bit=8):
    """
    權重重新排列：從 (kernel, ch_in, ch_out) 轉為 (ch_in, kernel, ch_out) 並由右至左打包。
    
    :param source_data: 來源權重 txt 路徑
    :param target_data: 輸出重排後 txt 路徑
    :param k_size: Kernel 數量
    :param ch_in: Input Channel 數量
    :param ch_out: Output Channel 數量
    :param word_size: 輸出匯流排位元寬度 (預設 32)
    :param data_bit: 權重資料位元寬度 (預設 8)
    """
    slots_per_word = word_size // data_bit
    chars_per_data = data_bit // 4
    
    os.makedirs(os.path.dirname(target_data), exist_ok=True)

    try:
        # 1. 讀取並解析原始資料
        with open(source_data, 'r', encoding='utf-8') as f:
            raw_text = "".join(f.read().split())
            raw_elements = [raw_text[i:i+chars_per_data] for i in range(0, len(raw_text), chars_per_data)]

        expected_total = k_size * ch_in * ch_out
        if len(raw_elements) < expected_total:
            print(f"Error: 資料量不足。預期 {expected_total}, 實際 {len(raw_elements)}")
            return
        
        # 2. 建立三維矩陣：[kernel][ch_in][ch_out]
        # 模仿原始存放順序
        weight_3d = []
        idx = 0
        for k in range(k_size):
            layer_ch_in = []
            for ci in range(ch_in):
                layer_ch_out = raw_elements[idx : idx + ch_out]
                layer_ch_in.append(layer_ch_out)
                idx += ch_out
            weight_3d.append(layer_ch_in)

        # 3. 轉置資料流：改為 (ch_in, kernel, ch_out)
        target_stream = []
        for ci in range(ch_in):
            for k in range(k_size):
                for co in range(ch_out):
                    target_stream.append(weight_3d[k][ci][co])

        # 4. 打包寫入 (由右往左擺放)
        with open(target_data, 'w', encoding='utf-8') as f_out:
            for i in range(0, len(target_stream), slots_per_word):
                chunk = target_stream[i : i + slots_per_word]
                
                # 最後不足一字元寬度則補零
                while len(chunk) < slots_per_word:
                    chunk.append("0" * chars_per_data)
                
                # 由右至左：reversed 拼接
                packed_word = "".join(reversed(chunk))
                f_out.write(packed_word + '\n')

        print(f"權重轉換成功！順序已轉為 (ch_in, kernel, ch_out)")
        print(f"輸出路徑: {target_data}")

    except Exception as e:
        print(f"發生錯誤: {e}")

# --- 測試範例 ---
# 假設資料 16 筆，k=2, ch_in=2, ch_out=4
# rearrange_weight("weight.txt", "weight_rearrange.txt", k_size=2, ch_in=2, ch_out=4)

In [46]:
# conv1
rearrange_weight("./Data_NPU/Data_ori_BtoH/conv1.txt", "./Data_NPU/model/conv1/ker.txt", k_size=4, ch_in=1, ch_out=4, word_size=32, data_bit=8)
rearrange_weight("./Data_NPU/Data_ori_BtoH/conv1_bias.txt", "./Data_NPU/model/conv1/bias.txt", k_size=1, ch_in=1, ch_out=4, word_size=32, data_bit=32)

權重轉換成功！順序已轉為 (ch_in, kernel, ch_out)
輸出路徑: ./Data_NPU/model/conv1/ker.txt
權重轉換成功！順序已轉為 (ch_in, kernel, ch_out)
輸出路徑: ./Data_NPU/model/conv1/bias.txt
